# LocStab: reproducible community benchmark

This is the corrected NUMBA benchmark.  Shared experiment logic lives in
`BenchmarkNumbaExperiments.py`, which prevents the LocPop and LocStab protocols
from drifting apart.

The run uses fixed data seeds, ten paired permutations, standardized vector
features, disjoint friendship/enmity relations, correctly named adjusted Rand
index (ARI), same-run warm starts, convergence diagnostics, and explicit
algorithm parameters.  It writes both raw and summarized CSVs and regenerates
individual plus four-panel summary plots.

For community detection, the singleton initialization is omitted on Cora for
tractability and the predicted-$k$ initialization is omitted on Jazz because
that dataset has no reference $k$.  A predicted-$k$ Cora start has capacity 50,
while a Leiden start retains any larger capacity required by its initial labels.
All other datasets allow up to $n$ labels and therefore include singleton-
creation moves whenever an empty label is available.  Random-25 uses 25
connected ten-node Erdős--Rényi blocks ($p=0.40$), independent cross-block
edges with probability $q=0.001$, and one random bridge between consecutive
blocks on a ring, ensuring a connected but community-structured graph.

Expected outputs:

- `csv/StableCommunity/runs.csv`
- `csv/StableCommunity/results.csv`
- `csv/StableCommunity/preprocessing.csv`
- `csv/StableCommunity/data-diagnostics.csv`
- `csv/StableCommunity/dataset-0.csv`, `dataset-1.csv`, `dataset-2.csv`
- `figures/StableCommunity/*.png`


In [1]:
from importlib.metadata import version
from BenchmarkNumbaExperiments import run_community_experiment, DATA_SEED, THRESHOLDS, DOMAINS, RANDOM25_WITHIN_P, RANDOM25_BETWEEN_P

REPETITIONS = 10
LOCAL_STABLE = True

print("Data seed:", DATA_SEED)
print("Repetitions:", REPETITIONS)
print("Thresholds:", THRESHOLDS)
print("Domains:", DOMAINS)

print("Random-25 probabilities (within, between):", RANDOM25_WITHIN_P, RANDOM25_BETWEEN_P)
for package in ["numpy", "scipy", "pandas", "scikit-learn", "networkx", "numba", "matplotlib"]:
    print(f"{package}={version(package)}")


Data seed: 20260817
Repetitions: 10
Thresholds: ((0.2, 0.2), (0.25, 0.35), (0.4, 0.4))
Domains: ('B', 'AF', 'AE')
Random-25 probabilities (within, between): 0.4 0.001
numpy=2.5.2
scipy=1.18.0
pandas=3.0.5
scikit-learn=1.9.0
networkx=3.6.1
numba=0.67.0
matplotlib=3.11.1


## Execute the complete experiment

This cell performs the full production run and overwrites the corresponding
CSV and figure artifacts.  In particular, Cora's all-pairs preprocessing and
large-instance heuristic runs can take several minutes.  Do not interrupt the
kernel while files are being written.


In [2]:
summary = run_community_experiment(local_stable=LOCAL_STABLE, repetitions=REPETITIONS)
summary

,Method,Dataset,Preference,Initialization,Beta Friend,Beta Enemy,Repetitions,Adjusted Rand Index,Adjusted Rand Index SD,Modularity,...,Seconds,Seconds SD,Moves,Moves SD,Converged,Converged SD,Final Coalitions,Final Coalitions SD,Initial Adjusted Rand Index,Initial Adjusted Rand Index SD
0,Louvain,Karate Club,-,-,NaN,NaN,10,0.438855,0.034855,0.439534,...,0.006930,0.001282,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Leiden,Karate Club,-,-,NaN,NaN,10,0.426886,0.037952,0.440923,...,0.014745,0.003025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LocStab,Karate Club,B,Ld,0.2,0.2,10,0.132099,0.002195,0.294619,...,0.049418,0.145341,14.2,0.400000,1.0,0.0,16.2,0.600000,0.373084,0.018940
3,LocStab,Karate Club,B,S,0.2,0.2,10,0.096005,0.008734,0.250469,...,0.000972,0.000208,17.6,1.496663,1.0,0.0,18.8,0.400000,0.000000,0.000000
4,LocStab,Karate Club,B,P,0.2,0.2,10,0.111214,0.011134,0.267546,...,0.000947,0.000159,27.8,1.833030,1.0,0.0,18.4,0.800000,0.010145,0.022421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,LocStab,Random-25,AF,S,0.4,0.4,10,0.139864,0.014403,0.517648,...,0.037006,0.001469,244.6,5.180734,1.0,0.0,11.9,0.943398,0.000000,0.000000
94,LocStab,Random-25,AF,P,0.4,0.4,10,0.179911,0.017020,0.573035,...,0.025893,0.001887,233.8,11.311941,1.0,0.0,11.1,0.830662,0.003552,0.003398
95,LocStab,Random-25,AE,Ld,0.4,0.4,10,0.440245,0.026411,0.671547,...,0.014117,0.000779,108.7,10.198529,1.0,0.0,16.9,0.830662,0.498783,0.038742
96,LocStab,Random-25,AE,S,0.4,0.4,10,0.355994,0.022347,0.565228,...,0.036934,0.000665,244.5,6.484597,1.0,0.0,21.3,1.268858,0.000000,0.000000


## Verify convergence and output coverage

In [3]:
heuristic = summary[summary["Method"] == "LocStab"]
print("Summary rows:", len(summary))
print("Datasets:", sorted(summary["Dataset"].unique()))
print("Minimum convergence rate:", heuristic["Converged"].min())
print("Maximum recorded moves:", heuristic["Moves"].max())

if not (heuristic["Converged"] == 1.0).all():
    display(heuristic[heuristic["Converged"] < 1.0])
    raise RuntimeError("At least one run reached the move cap; do not use the outputs without investigation.")

display(summary.sort_values(["Dataset", "Method", "Initialization", "Preference"]).head(20))
print("Artifacts written under csv/StableCommunity and figures/StableCommunity")


Summary rows: 98
Datasets: ['Cora', 'Jazz', 'Karate Club', 'Random-25']
Minimum convergence rate: 1.0
Maximum recorded moves: 3538.4


,Method,Dataset,Preference,Initialization,Beta Friend,Beta Enemy,Repetitions,Adjusted Rand Index,Adjusted Rand Index SD,Modularity,...,Seconds,Seconds SD,Moves,Moves SD,Converged,Converged SD,Final Coalitions,Final Coalitions SD,Initial Adjusted Rand Index,Initial Adjusted Rand Index SD
30,Leiden,Cora,-,-,NaN,NaN,10,0.143650,0.013804,0.798567,...,4.310795,0.277992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,LocStab,Cora,AE,Ld,0.20,0.20,10,0.121568,0.008968,0.641805,...,0.848813,0.063190,1183.3,33.277770,1.0,0.0,127.4,2.690725,4.876000e-01,0.032440
41,LocStab,Cora,AE,Ld,0.25,0.35,10,0.118939,0.003867,0.490688,...,1.063657,0.101363,1656.2,80.296700,1.0,0.0,127.4,2.690725,1.779506e-01,0.017827
47,LocStab,Cora,AE,Ld,0.40,0.40,10,0.113220,0.018474,0.437637,...,1.120273,0.131189,1646.2,128.309626,1.0,0.0,124.8,1.777639,1.442237e-01,0.030656
33,LocStab,Cora,AF,Ld,0.20,0.20,10,0.200907,0.004695,0.701519,...,0.544255,0.050214,648.8,56.060325,1.0,0.0,127.4,2.690725,5.598065e-01,0.037807
39,LocStab,Cora,AF,Ld,0.25,0.35,10,0.044636,0.001600,0.388505,...,1.009990,0.165364,1494.6,63.599057,1.0,0.0,127.4,2.690725,8.613842e-02,0.011011
45,LocStab,Cora,AF,Ld,0.40,0.40,10,-0.001436,0.000040,0.145689,...,1.329890,0.112689,2075.5,30.621071,1.0,0.0,113.6,1.019804,2.108803e-02,0.002467
31,LocStab,Cora,B,Ld,0.20,0.20,10,0.163914,0.001471,0.637506,...,0.999442,0.071156,1393.4,30.020660,1.0,0.0,127.4,2.690725,3.770794e-01,0.027852
37,LocStab,Cora,B,Ld,0.25,0.35,10,0.037371,0.000264,0.381144,...,1.035128,0.090484,1608.4,48.957533,1.0,0.0,127.4,2.690725,7.667137e-02,0.008773
43,LocStab,Cora,B,Ld,0.40,0.40,10,-0.001540,0.000007,0.145597,...,1.417522,0.084746,2079.0,30.977411,1.0,0.0,118.0,0.447214,2.082321e-02,0.002402


Artifacts written under csv/StableCommunity and figures/StableCommunity
